In [5]:
import pandas as pd
import numpy as np

# Set a random seed so the generated random numbers are consistent every time you run it
np.random.seed(42)

# Create a date range for 90 days (July 1, 2026 to September 30, 2026)
date_range = pd.date_range(start="2026-07-01", end="2026-09-30")

# Build a dictionary containing simulated website metrics
data = {
    "date": date_range,
    "pageviews": np.random.randint(5000, 15000, size=len(date_range)),          # Random daily pageviews between 5k and 15k
    "unique_visitors": np.random.randint(1200, 4000, size=len(date_range)),    # Random unique visitors between 1.2k and 4k
    "bounce_rate": np.round(np.random.uniform(0.30, 0.60, size=len(date_range)), 2), # Random bounce rate between 30% and 60%
    "marketing_spend": np.round(np.random.uniform(200, 800, size=len(date_range)), 2), # Random daily ad spend between $200 and $800
}

# Convert the dictionary into a structured Pandas DataFrame (table format)
df = pd.DataFrame(data)

# Simulate actual sales based on pageviews, ad spend, and a small amount of random market noise
df["actual_sales"] = np.round(
    (df["pageviews"] * 0.25) + (df["marketing_spend"] * 1.5) + np.random.normal(0, 200, size=len(date_range)), 2
)

# Print a confirmation message and display the first 5 rows of your data table
print("Dataset generated successfully!")
df.head()

Dataset generated successfully!


,date,pageviews,unique_visitors,bounce_rate,marketing_spend,actual_sales
0,2026-07-01,12270,2695,0.59,501.58,3895.33
1,2026-07-02,5860,2362,0.59,546.14,2614.48
2,2026-07-03,10390,2722,0.38,495.51,3075.40
3,2026-07-04,10191,3840,0.45,317.15,3392.22
4,2026-07-05,10734,1591,0.39,633.47,3487.56


In [6]:
import urllib
from sqlalchemy import create_engine, URL

# Your Azure server and database details
server = "barathserver.database.windows.net"
database = "TrafficDB"  
username = "barathadmin"
password = "Barath@1903"

# Using ODBC Driver 18 which is installed on your system
connection_url = URL.create(
    "mssql+pyodbc",
    username=username,
    password=password,
    host=server,
    database=database,
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "TrustServerCertificate": "yes",
    },
)

# Create the database engine
engine = create_engine(connection_url)

# Upload the DataFrame directly to your Azure SQL table
df.to_sql("daily_traffic_sales", con=engine, if_exists="replace", index=False)

print("Success! Data pushed from Python to Azure SQL Database!")

Success! Data pushed from Python to Azure SQL Database!


In [7]:
import pandas as pd
from sqlalchemy import create_engine, URL

# ==========================================
# PHASE 1: ESTABLISH SECURE CLOUD CONNECTION
# ==========================================
# Build the secure connection parameters using SQLAlchemy's URL builder 
# and Driver 18 for your Azure SQL instance.
connection_url = URL.create(
    "mssql+pyodbc",
    username="barathadmin",  # Replace with your Azure SQL admin username
    password="Barath@1903",  # Replace with your Azure SQL admin password
    host="barathserver.database.windows.net",
    database="TrafficDB",
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "TrustServerCertificate": "yes",  # Bypass local SSL certificate validation for cloud connection
    },
)

# Initialize the SQLAlchemy database engine connection pool
engine = create_engine(connection_url)


# ==========================================
# PHASE 2: WRITE & EXECUTE THE SQL QUERY
# ==========================================
# Define a standard SQL query to grab the top 10 most recent rows 
# from the 'daily_traffic_sales' table we uploaded earlier.
query = """
    SELECT TOP 10 
        date, 
        pageviews, 
        unique_visitors, 
        bounce_rate, 
        marketing_spend, 
        actual_sales 
    FROM daily_traffic_sales 
    ORDER BY date DESC;
"""

# Execute the query and load the results directly into a Pandas DataFrame
df_from_azure = pd.read_sql(query, con=engine)


# ==========================================
# PHASE 3: VERIFY AND DISPLAY THE RESULTS
# ==========================================
# Print confirmation and display the retrieved data table in your notebook
print("Success! Data successfully retrieved from Azure SQL Database:\n")
print(df_from_azure)

Success! Data successfully retrieved from Azure SQL Database:

        date  pageviews  unique_visitors  bounce_rate  marketing_spend  \
0 2026-09-30       7695             3053         0.54           596.12   
1 2026-09-29       7027             2336         0.49           628.76   
2 2026-09-28      10463             2695         0.57           529.54   
3 2026-09-27       7568             2831         0.42           350.15   
4 2026-09-26      13006             3198         0.59           293.86   
5 2026-09-25       5064             1863         0.37           303.98   
6 2026-09-24       7062             1883         0.38           445.37   
7 2026-09-23      10276             2457         0.41           617.58   
8 2026-09-22      12208             1517         0.33           701.59   
9 2026-09-21      13684             1848         0.47           777.59   

   actual_sales  
0       2817.02  
1       2639.43  
2       3724.07  
3       2009.81  
4       3964.31  
5       1589.1

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sqlalchemy import create_engine, URL

# ==========================================
# PHASE 1: LOAD FULL DATASET FROM AZURE SQL
# ==========================================
# Reconnect to Azure and pull the entire table into a working dataframe for ML
connection_url = URL.create(
    "mssql+pyodbc",
    username="barathadmin",
    password="Barath@1903",
    host="barathserver.database.windows.net",
    database="TrafficDB",
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "TrustServerCertificate": "yes",
    },
)
engine = create_engine(connection_url)

# Query all rows and columns needed for prediction
query = "SELECT pageviews, unique_visitors, bounce_rate, marketing_spend, actual_sales FROM daily_traffic_sales;"
df_ml = pd.read_sql(query, con=engine)


# ==========================================
# PHASE 2: PREPARE FEATURES & TARGET VARIABLES
# ==========================================
# X contains our independent variables (features used to make predictions)
X = df_ml[['pageviews', 'unique_visitors', 'bounce_rate', 'marketing_spend']]

# y contains our dependent variable (the target value we want to predict: sales)
y = df_ml['actual_sales']

# Split the data: 80% for training the model, 20% for testing its accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# ==========================================
# PHASE 3: TRAIN THE MACHINE LEARNING MODEL
# ==========================================
# Initialize the Linear Regression algorithm
model = LinearRegression()

# Train (fit) the model using the training data
model.fit(X_train, y_train)


# ==========================================
# PHASE 4: EVALUATE MODEL PERFORMANCE
# ==========================================
# Predict sales on the unseen test data
y_pred = model.predict(X_test)

# Calculate performance metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("Machine Learning Model Training Complete!\n")
print(f"R-squared (Accuracy Score): {r2:.4f} (Close to 1.0 is exceptional)")
print(f"Mean Squared Error: {mse:.4f}\n")

# Display how much weight/impact each feature has on actual sales
coefficients = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
print("Feature Impact Analysis:")
print(coefficients)

Machine Learning Model Training Complete!

R-squared (Accuracy Score): 0.8578 (Close to 1.0 is exceptional)
Mean Squared Error: 52550.8798

Feature Impact Analysis:
                 Coefficient
pageviews           0.257413
unique_visitors     0.008541
bounce_rate       605.061953
marketing_spend     1.604879


In [9]:
import pandas as pd
from sqlalchemy import create_engine, URL

# ==========================================
# PHASE 1: CONNECT TO AZURE SQL
# ==========================================
# Set up your connection string using SQLAlchemy and Driver 18
connection_url = URL.create(
    "mssql+pyodbc",
    username="barathadmin",  # Replace with your Azure SQL admin username
    password="Barath@1903",  # Replace with your Azure SQL admin password
    host="barathserver.database.windows.net",
    database="TrafficDB",
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "TrustServerCertificate": "yes",
    },
)

engine = create_engine(connection_url)


# ==========================================
# PHASE 2: QUERY 1 - 7-DAY MOVING AVERAGE SALES TREND
# ==========================================
# This uses a SQL window function (AVG() OVER) to smooth out daily sales noise
query_moving_avg = """
    SELECT 
        date, 
        actual_sales,
        AVG(actual_sales) OVER (
            ORDER BY date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7day_avg_sales
    FROM daily_traffic_sales
    ORDER BY date;
"""

df_moving_avg = pd.read_sql(query_moving_avg, con=engine)

print("--- 7-Day Moving Average Sales Trend (First 10 Rows) ---")
print(df_moving_avg.head(10))
print("\n" + "="*50 + "\n")


# ==========================================
# PHASE 3: QUERY 2 - TOP 5 PERFORMANCE DAYS
# ==========================================
# This query calculates marketing efficiency (sales generated per dollar spent) 
# and returns your top 5 highest sales days
query_top_days = """
    SELECT TOP 5 
        date, 
        pageviews, 
        marketing_spend, 
        actual_sales,
        ROUND(actual_sales / NULLIF(marketing_spend, 0), 2) AS sales_per_dollar_spent
    FROM daily_traffic_sales
    ORDER BY actual_sales DESC;
"""

df_top_days = pd.read_sql(query_top_days, con=engine)

print("--- Top 5 Performance Days ---")
print(df_top_days)

--- 7-Day Moving Average Sales Trend (First 10 Rows) ---
        date  actual_sales  rolling_7day_avg_sales
0 2026-07-01       3895.33             3895.330000
1 2026-07-02       2614.48             3254.905000
2 2026-07-03       3075.40             3195.070000
3 2026-07-04       3392.22             3244.357500
4 2026-07-05       3487.56             3292.998000
5 2026-07-06       3112.96             3262.991667
6 2026-07-07       1640.91             3031.265714
7 2026-07-08       3611.82             2990.764286
8 2026-07-09       3100.02             3060.127143
9 2026-07-10       4290.40             3233.698571


--- Top 5 Performance Days ---
        date  pageviews  marketing_spend  actual_sales  sales_per_dollar_spent
0 2026-08-31      14692           693.56       4884.13                    7.04
1 2026-08-11      13792           720.24       4818.87                    6.69
2 2026-09-16      14268           677.11       4743.43                    7.01
3 2026-09-21      13684          